### Phase 2 | Ranking Algorithm Experiment with Optuna Hyperparameter Tuning


Pada notebook ini dilakukan eksperimen pencarian hyperparameter terbaik (*Hyperparameter Tuning*) untuk LightGBM Ranker, XGBoost Ranker, SVM Ranker, dan Random Forest menggunakan **Optuna** dan **GroupKFold Cross-Validation**.


#### Step 1 | Setup Library & Environment


In [1]:
import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost
import optuna
from dotenv import load_dotenv
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import LinearSVC
import lightgbm as lgb
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from IPython.display import display
except ImportError:
    display = print

load_dotenv()

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(mlflow_uri)

mlflow.set_experiment("Phase_2_Ranking_Algorithm_Tuning_Experiment")

print(f"MLflow Connected to: {mlflow_uri}")
print(f"Optuna Version: {optuna.__version__}")


MLflow Connected to: https://mlflow.smbgarasibmw.my.id/
Optuna Version: 4.9.0


#### Step 2 | Load Data & Validasi Query Group


In [2]:
df = pd.read_csv("dataset/Dataset_EDA_TFIDF.csv")

TARGET_COL = "priority_score"
QID_COL = "project_id"

df = df.sort_values(by=QID_COL).reset_index(drop=True)

print(f"Total Sampel Requirement : {len(df)}")
print(f"Total Jumlah Proyek (QID) : {df[QID_COL].nunique()}")
print(f"Daftar Proyek             : {df[QID_COL].unique()}")

df.head()


Total Sampel Requirement : 698
Total Jumlah Proyek (QID) : 15
Daftar Proyek             : ['P1' 'P10' 'P11' 'P12' 'P13' 'P14' 'P15' 'P2' 'P3' 'P4' 'P5' 'P6' 'P7'
 'P8' 'P9']


,id,project_id,type,value,effort,risk,stakeholder_priority,priority_score,rank,cleaned_requirement_text,...,tfidf_work,tfidf_workflow,tfidf_worklist,tfidf_workshop,tfidf_workstation,tfidf_xml,tfidf_year,tfidf_yet,tfidf_zero,tfidf_zipcode
0,REQ-01,P1,FR,-0.848439,-1.015393,-1.393797,-0.8401,1.3,18,head admin finance edit delete spare part cate...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,REQ-505,P1,NFR,-0.848439,0.165847,-0.550024,-0.8401,1.4,16,cosign note record date time signature,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,REQ-506,P1,NFR,0.444597,0.165847,1.981294,-0.8401,1.2,25,record display identity addended correct note ...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,REQ-507,P1,NFR,-0.848439,0.165847,1.981294,-0.8401,0.8,33,record patient specific instruction educationa...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,REQ-508,P1,NFR,-0.848439,0.165847,1.981294,-0.8401,0.8,33,record date stamp prescription event initial c...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Step 3 | Group Based Data Splitting


In [3]:
# 1. Pisahkan Query ID dan Target
qids = df[QID_COL]
y = (df[TARGET_COL] * 10).astype(int)  # Ubah float (1.5) jadi int (15) untuk Ranker

# 2. Hapus kolom metadata & target dari matriks fitur X
COLS_TO_DROP = [
    "id",                       # ID requirement
    "type",                     # Jenis requirement (FR/NFR)
    "rank",                     # WAJIB DROP: Jangan sampai terbawa (data leakage)
    "cleaned_requirement_text", # Teks mentah
    TARGET_COL,                 # priority_score
    QID_COL                     # project_id
]

X = df.drop(columns=COLS_TO_DROP)

# 3. GroupShuffleSplit (80% Train, 20% Test berdasarkan kelompok project_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=qids))

# 4. Potong Data Train & Test
X_train, y_train, qids_train = X.iloc[train_idx].copy(), y.iloc[train_idx].copy(), qids.iloc[train_idx].copy()
X_test, y_test, qids_test = X.iloc[test_idx].copy(), y.iloc[test_idx].copy(), qids.iloc[test_idx].copy()

# 5. Urutkan runtut berdasarkan project_id
train_order = np.argsort(qids_train.values)
X_train = X_train.iloc[train_order].reset_index(drop=True)
y_train = y_train.iloc[train_order].reset_index(drop=True)
qids_train = qids_train.iloc[train_order].reset_index(drop=True)

test_order = np.argsort(qids_test.values)
X_test = X_test.iloc[test_order].reset_index(drop=True)
y_test = y_test.iloc[test_order].reset_index(drop=True)
qids_test = qids_test.iloc[test_order].reset_index(drop=True)

# 6. Hitung struktur ukuran grup per proyek
group_train = qids_train.value_counts(sort=False).values
group_test = qids_test.value_counts(sort=False).values

print(f"Data Train : {len(X_train)} requirement dari {len(group_train)} proyek")
print(f"Data Test  : {len(X_test)} requirement dari {len(group_test)} proyek")
print(f"Jumlah Fitur X : {X_train.shape[1]} (Domain features [value, effort, risk, stakeholder] + TF-IDF)")


Data Train : 453 requirement dari 12 proyek
Data Test  : 245 requirement dari 3 proyek
Jumlah Fitur X : 1431 (Domain features [value, effort, risk, stakeholder] + TF-IDF)


#### Step 4 | Evaluasi Rangking (NDCG, MAP, Spearman, Kendall)


In [4]:
def compute_map(y_true, y_pred, threshold=None):
    """Menghitung Mean Average Precision (MAP) per kelompok."""
    if threshold is None:
        threshold = np.median(y_true)
    
    binary_true = (y_true >= threshold).astype(int)
    if binary_true.sum() == 0:
        return 0.0
    
    # Sort berdasarkan prediksi berurutan menurun (descending)
    order = np.argsort(-y_pred)
    binary_true = binary_true[order]
    
    precisions = []
    n_relevant = 0
    for i, rel in enumerate(binary_true):
        if rel == 1:
            n_relevant += 1
            precisions.append(n_relevant / (i + 1))
            
    return np.mean(precisions) if precisions else 0.0

def evaluate_ranking_metrics(y_true, y_pred, qids, k=5):
    """Evaluasi korelasi urutan dan mutu ranking secara rata-rata di seluruh query group."""
    spearman_list, kendall_list, ndcg_list, map_list = [], [], [], []
    
    for qid in np.unique(qids):
        mask = (qids == qid)
        yt = y_true[mask].values
        yp = y_pred[mask]
        
        if len(yt) > 1:
            # Spearman & Kendall Tau
            rh, _ = spearmanr(yt, yp)
            kt, _ = kendalltau(yt, yp)
            if not np.isnan(rh): spearman_list.append(rh)
            if not np.isnan(kt): kendall_list.append(kt)
            
            # NDCG_K
            ndcg = ndcg_score([yt], [yp], k=k)
            ndcg_list.append(ndcg)
            
            # MAP
            map_val = compute_map(yt, yp)
            map_list.append(map_val)
            
    return {
        "Spearman": np.mean(spearman_list) if spearman_list else 0.0,
        "Kendall": np.mean(kendall_list) if kendall_list else 0.0,
        f"NDCG_{k}": np.mean(ndcg_list) if ndcg_list else 0.0,
        "MAP": np.mean(map_list) if map_list else 0.0
    }


#### Step 5 | Pairwise Data Construction (SVMRank)


In [5]:
def create_pairwise_data(X_data, y_data, qids_data):
    """Mengubah dataset pointwise menjadi pasangan perbedaan fitur (Pairwise transformation untuk SVMRank)."""
    X_pairs, y_pairs = [], []
    
    for qid in np.unique(qids_data):
        idx = np.where(qids_data == qid)[0]
        X_group = X_data.iloc[idx].values
        y_group = y_data.iloc[idx].values
        
        n_samples = len(idx)
        for i in range(n_samples):
            for j in range(i + 1, n_samples):
                if y_group[i] != y_group[j]:
                    # Selisih fitur
                    diff = X_group[i] - X_group[j]
                    label = 1 if y_group[i] > y_group[j] else -1
                    
                    X_pairs.append(diff)
                    y_pairs.append(label)
                    
                    # Tambahkan simetri sebaliknya untuk menyeimbangkan kelas (+1 dan -1)
                    X_pairs.append(-diff)
                    y_pairs.append(-label)
                    
    return np.array(X_pairs), np.array(y_pairs)

print("Helper Pairwise SVMRank siap digunakan.")


Helper Pairwise SVMRank siap digunakan.


#### Step 6 | Training 4 Algoritma dengan Optuna Hyperparameter Tuning


In [6]:
import joblib

# Pre-split data untuk 3-Fold GroupKFold agar proses tuning lebih cepat
gkf = GroupKFold(n_splits=3)
cv_folds = []
svm_cv_folds = []

for tr_idx, val_idx in gkf.split(X_train, y_train, groups=qids_train):
    X_tr, y_tr, qids_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx], qids_train.iloc[tr_idx]
    X_va, y_va, qids_va = X_train.iloc[val_idx], y_train.iloc[val_idx], qids_train.iloc[val_idx]
    group_tr = qids_tr.value_counts(sort=False).values
    
    cv_folds.append((X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va))
    
    # Precompute pairwise data untuk SVMRank
    X_tr_p, y_tr_p = create_pairwise_data(X_tr, y_tr, qids_tr)
    svm_cv_folds.append((X_tr_p, y_tr_p, X_va, y_va, qids_va))

results_summary = {}
trained_models = {}
best_params_all = {}
df_preds = df.iloc[test_idx][["id", "project_id", "priority_score"]].iloc[test_order].copy().reset_index(drop=True)

print("Pre-split GroupKFold CV folds selesai dikonstruksi.")


Pre-split GroupKFold CV folds selesai dikonstruksi.


##### Step 6.1 | Optuna Tuning & Training LightGBM Ranker


In [7]:
model_name = "LightGBM_Ranker_Tuned"

def objective_lgb(trial):
    params = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1
    }
    
    scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds:
        model = lgb.LGBMRanker(**params)
        model.fit(X_tr, y_tr, group=group_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        scores.append(metrics["NDCG_5"])
        
    return np.mean(scores)

with mlflow.start_run(run_name=model_name):
    print(f"=================== Optuna Tuning {model_name} ===================")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective_lgb, n_trials=15)
    
    best_params = study.best_params
    best_params.update({"objective": "lambdarank", "metric": "ndcg", "random_state": 42, "verbose": -1, "n_jobs": -1})
    print(f"Best CV NDCG_5 Score : {study.best_value:.4f}")
    print(f"Best Hyperparameters  : {study.best_params}\n")
    
    # Train Best Model on Full X_train
    best_model = lgb.LGBMRanker(**best_params)
    best_model.fit(X_train, y_train, group=group_train)
    preds = best_model.predict(X_test)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = best_model
    best_params_all[model_name] = study.best_params
    
    mlflow.log_params(best_params)
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> Test {m_name}: {m_val:.4f}")
        
    mlflow.lightgbm.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
    print()


=================== Optuna Tuning LightGBM_Ranker_Tuned ===================


Best CV NDCG_5 Score : 0.9743
Best Hyperparameters  : {'n_estimators': 50, 'learning_rate': 0.08115595675970502, 'num_leaves': 27, 'max_depth': 6, 'subsample': 0.7246844304357644, 'colsample_bytree': 0.8080272084711243}

-> Test Spearman: 0.8745


2026/08/13 12:59:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


-> Test Kendall: 0.7496
-> Test NDCG_5: 0.9887
-> Test MAP: 0.9508


2026/08/13 12:59:48 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run LightGBM_Ranker_Tuned at: https://mlflow.smbgarasibmw.my.id/#/experiments/6/runs/415349a8d75e4846b2a0d8b2d1c40ac0
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/6


##### Step 6.2 | Optuna Tuning & Training XGBoost Ranker


In [8]:
model_name = "XGBoost_Ranker_Tuned"

def objective_xgb(trial):
    params = {
        "objective": "rank:ndcg",
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "n_jobs": -1
    }
    
    scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds:
        model = xgb.XGBRanker(**params)
        model.fit(X_tr, y_tr, group=group_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        scores.append(metrics["NDCG_5"])
        
    return np.mean(scores)

with mlflow.start_run(run_name=model_name):
    print(f"=================== Optuna Tuning {model_name} ===================")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective_xgb, n_trials=15)
    
    best_params = study.best_params
    best_params.update({"objective": "rank:ndcg", "random_state": 42, "n_jobs": -1})
    print(f"Best CV NDCG_5 Score : {study.best_value:.4f}")
    print(f"Best Hyperparameters  : {study.best_params}\n")
    
    # Train Best Model on Full X_train
    best_model = xgb.XGBRanker(**best_params)
    best_model.fit(X_train, y_train, group=group_train)
    preds = best_model.predict(X_test)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = best_model
    best_params_all[model_name] = study.best_params
    
    mlflow.log_params(best_params)
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> Test {m_name}: {m_val:.4f}")
        
    mlflow.xgboost.log_model(best_model, artifact_path="model")
    print()


=================== Optuna Tuning XGBoost_Ranker_Tuned ===================


Best CV NDCG_5 Score : 0.9705
Best Hyperparameters  : {'n_estimators': 100, 'learning_rate': 0.048965525907765685, 'max_depth': 5, 'subsample': 0.602130509414732, 'colsample_bytree': 0.9181971827644655}



-> Test Spearman: 0.8371
-> Test Kendall: 0.6922
-> Test NDCG_5: 0.9893


2026/08/13 13:00:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


-> Test MAP: 0.9422



🏃 View run XGBoost_Ranker_Tuned at: https://mlflow.smbgarasibmw.my.id/#/experiments/6/runs/77a9ec1d65724cd3ba564c5e910a2827
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/6


##### Step 6.3 | Optuna Tuning & Training SVMRank


In [9]:
model_name = "SVMRank_Tuned"

def objective_svm(trial):
    C_val = trial.suggest_float("C", 1e-3, 10.0, log=True)
    loss_val = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])
    
    scores = []
    for X_tr_p, y_tr_p, X_va, y_va, qids_va in svm_cv_folds:
        model = LinearSVC(C=C_val, loss=loss_val, random_state=42, max_iter=5000, dual="auto")
        model.fit(X_tr_p, y_tr_p)
        preds = model.decision_function(X_va.values)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        scores.append(metrics["NDCG_5"])
        
    return np.mean(scores)

with mlflow.start_run(run_name=model_name):
    print(f"=================== Optuna Tuning {model_name} ===================")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective_svm, n_trials=10)
    
    best_params = study.best_params
    best_params.update({"random_state": 42, "max_iter": 5000, "dual": "auto"})
    print(f"Best CV NDCG_5 Score : {study.best_value:.4f}")
    print(f"Best Hyperparameters  : {study.best_params}\n")
    
    # Train Best Model on Full Pairwise Training Data
    X_train_p, y_train_p = create_pairwise_data(X_train, y_train, qids_train)
    best_model = LinearSVC(**best_params)
    best_model.fit(X_train_p, y_train_p)
    preds = best_model.decision_function(X_test.values)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = best_model
    best_params_all[model_name] = study.best_params
    
    mlflow.log_params(best_params)
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> Test {m_name}: {m_val:.4f}")
        
    mlflow.sklearn.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
    print()


=================== Optuna Tuning SVMRank_Tuned ===================


C:\Users\EdselSpth\AppData\Roaming\Python\Python311\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


C:\Users\EdselSpth\AppData\Roaming\Python\Python311\site-packages\sklearn\svm\_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best CV NDCG_5 Score : 1.0000
Best Hyperparameters  : {'C': 0.03148911647956861, 'loss': 'hinge'}



-> Test Spearman: 0.9900
-> Test Kendall: 0.9439
-> Test NDCG_5: 1.0000


2026/08/13 13:00:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


-> Test MAP: 1.0000


2026/08/13 13:00:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run SVMRank_Tuned at: https://mlflow.smbgarasibmw.my.id/#/experiments/6/runs/001a1fee5eb5408ba7dd819da959b90a
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/6


##### Step 6.4 | Optuna Tuning & Training Random Forest


In [10]:
model_name = "Random_Forest_Tuned"

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=25),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 4),
        "random_state": 42,
        "n_jobs": -1
    }
    
    scores = []
    for X_tr, y_tr, qids_tr, group_tr, X_va, y_va, qids_va in cv_folds:
        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        metrics = evaluate_ranking_metrics(y_va, preds, qids_va, k=5)
        scores.append(metrics["NDCG_5"])
        
    return np.mean(scores)

with mlflow.start_run(run_name=model_name):
    print(f"=================== Optuna Tuning {model_name} ===================")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective_rf, n_trials=15)
    
    best_params = study.best_params
    best_params.update({"random_state": 42, "n_jobs": -1})
    print(f"Best CV NDCG_5 Score : {study.best_value:.4f}")
    print(f"Best Hyperparameters  : {study.best_params}\n")
    
    # Train Best Model on Full X_train
    best_model = RandomForestRegressor(**best_params)
    best_model.fit(X_train, y_train)
    preds = best_model.predict(X_test)
    
    df_preds[f"pred_score_{model_name}"] = preds
    metrics = evaluate_ranking_metrics(y_test, preds, qids_test, k=5)
    results_summary[model_name] = metrics
    trained_models[model_name] = best_model
    best_params_all[model_name] = study.best_params
    
    mlflow.log_params(best_params)
    for m_name, m_val in metrics.items():
        mlflow.log_metric(m_name, m_val)
        print(f"-> Test {m_name}: {m_val:.4f}")
        
    mlflow.sklearn.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
    print()


=================== Optuna Tuning Random_Forest_Tuned ===================


Best CV NDCG_5 Score : 0.9709
Best Hyperparameters  : {'n_estimators': 125, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 4}



-> Test Spearman: 0.9601
-> Test Kendall: 0.8834
-> Test NDCG_5: 0.9906


2026/08/13 13:00:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


-> Test MAP: 0.9925


2026/08/13 13:00:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run Random_Forest_Tuned at: https://mlflow.smbgarasibmw.my.id/#/experiments/6/runs/d23ae4c6a1674f63b6139e509681ca6b
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/6


#### Step 7 | Export Results & Tuned Model Comparison


In [11]:
os.makedirs("results/models_tuned", exist_ok=True)

# 1. Simpan Ringkasan Perbandingan Metrik ke CSV
df_results = pd.DataFrame(results_summary).T
df_results.to_csv("results/phase_2_tuned_model_comparison.csv", index_label="Model")

print("=== HASIL EKSPERIMEN PHASE 2: TUNED RANKING ALGORITHM COMPARISON ===")
display(df_results)

# 2. Simpan Detail Prediksi Test Set ke CSV
df_preds.to_csv("results/phase_2_tuned_predictions.csv", index=False)

# 3. Simpan SEMUA model tuned ke folder results/models_tuned/
print("\n💾 Menyimpan semua file pickle (.pkl) model tuned ke 'results/models_tuned/':")
for name, m in trained_models.items():
    model_path = f"results/models_tuned/{name}.pkl"
    joblib.dump(m, model_path)
    print(f"   - Model '{name}' -> {model_path}")

# 4. Evaluasi & Simpan Model Terbaik ke results/best_ranking_model_tuned.pkl
best_model_name = df_results["NDCG_5"].idxmax()
best_score = df_results.loc[best_model_name, "NDCG_5"]

print(f"\n🏆 BEST TUNED RANKING ALGORITHM: {best_model_name}")
print(f"   Skor NDCG_5 Terbaik: {best_score:.4f}")
print(f"   Parameter Terbaik  : {best_params_all.get(best_model_name)}")

joblib.dump(trained_models[best_model_name], "results/best_ranking_model_tuned.pkl")

print("\n✅ Semua file hasil eksekusi berhasil disimpan di folder 'results/':")
print("   1. results/phase_2_tuned_model_comparison.csv")
print("   2. results/phase_2_tuned_predictions.csv")
print("   3. results/best_ranking_model_tuned.pkl")
print("   4. results/models_tuned/")


=== HASIL EKSPERIMEN PHASE 2: TUNED RANKING ALGORITHM COMPARISON ===


,Spearman,Kendall,NDCG_5,MAP
LightGBM_Ranker_Tuned,0.874465,0.749602,0.988703,0.950786
XGBoost_Ranker_Tuned,0.837132,0.692217,0.989332,0.942245
SVMRank_Tuned,0.989972,0.943941,1.000000,1.000000
Random_Forest_Tuned,0.960122,0.883443,0.990556,0.992481



💾 Menyimpan semua file pickle (.pkl) model tuned ke 'results/models_tuned/':
   - Model 'LightGBM_Ranker_Tuned' -> results/models_tuned/LightGBM_Ranker_Tuned.pkl
   - Model 'XGBoost_Ranker_Tuned' -> results/models_tuned/XGBoost_Ranker_Tuned.pkl
   - Model 'SVMRank_Tuned' -> results/models_tuned/SVMRank_Tuned.pkl
   - Model 'Random_Forest_Tuned' -> results/models_tuned/Random_Forest_Tuned.pkl

🏆 BEST TUNED RANKING ALGORITHM: SVMRank_Tuned
   Skor NDCG_5 Terbaik: 1.0000
   Parameter Terbaik  : {'C': 0.03148911647956861, 'loss': 'hinge'}

✅ Semua file hasil eksekusi berhasil disimpan di folder 'results/':
   1. results/phase_2_tuned_model_comparison.csv
   2. results/phase_2_tuned_predictions.csv
   3. results/best_ranking_model_tuned.pkl
   4. results/models_tuned/
